# GNN sur un dataset massif : Tox21
Puisque `ClinTox` était trop petit (1500 molécules) pour que le GNN devienne vraiment intelligent, nous passons à la vitesse supérieure !

Nous allons utiliser **Tox21** (Toxicology in the 21st Century) : une initiative majeure du gouvernement américain.
Ce dataset contient plus de **8 000 molécules**. 
Mieux encore : au lieu de prédire un simple "Toxique / Non Toxique", il prédit **12 cibles biologiques différentes** en même temps (ex: Est-ce que ça perturbe les hormones ? Est-ce que ça abîme l'ADN ?).

In [12]:
import logging
logging.getLogger("deepchem").setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')
from rdkit import RDLogger
RDLogger.DisableLog('rdApp.*') 

import torch
import deepchem as dc
import numpy as np

In [13]:
# 1. FEATURIZER
featurizer = dc.feat.ConvMolFeaturizer()

# 2. CHARGEMENT DE TOX21 (Au lieu de ClinTox)
print("Téléchargement et préparation de Tox21 (plus de 8000 graphes, cela prend une bonne minute...)...")
tasks, datasets, transformers = dc.molnet.load_tox21(featurizer=featurizer)
train_dataset, valid_dataset, test_dataset = datasets

print(f"\nTaille Entraînement: {len(train_dataset)} molécules")
print(f"Taille Test: {len(test_dataset)} molécules")
print("\nLes 12 types de toxicité étudiés (Récepteurs nucléaires, Stress cellulaire...) :\n", tasks)

Téléchargement et préparation de Tox21 (plus de 8000 graphes, cela prend une bonne minute...)...

Taille Entraînement: 6258 molécules
Taille Test: 783 molécules

Les 12 types de toxicité étudiés (Récepteurs nucléaires, Stress cellulaire...) :
 ['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


In [14]:
# 3. CRÉATION DU MODÈLE
# Le réseau aura 12 sorties indépendantes au lieu de 2 !
model = dc.models.GraphConvModel(
    n_tasks=len(tasks), 
    mode='classification',
    dropout=0.2
)

# 4. ENTRAÎNEMENT
# Attention, comme le dataset est 5 fois plus gros, chaque epoch sera plus longue.
print("Début de l'entraînement massif...")
model.fit(train_dataset, nb_epoch=10)
print("Entraînement terminé !")

Début de l'entraînement massif...
Entraînement terminé !


In [15]:
# 5. ÉVALUATION
metric = dc.metrics.Metric(dc.metrics.roc_auc_score)
train_scores = model.evaluate(train_dataset, [metric], transformers)
test_scores = model.evaluate(test_dataset, [metric], transformers)

print(f"Score moyen sur Entraînement : {train_scores['roc_auc_score']:.3f}")
print(f"Score moyen sur Test : {test_scores['roc_auc_score']:.3f}")

Score moyen sur Entraînement : 0.845
Score moyen sur Test : 0.684


In [16]:
# 6. TESTONS SUR LE CYANURE D'HYDROGÈNE
molecule = "C#N"
graphe_test = featurizer.featurize([molecule])
dataset_test = dc.data.NumpyDataset(X=graphe_test)

# La prédiction renvoie un tableau avec les 12 probabilités (une pour chaque cible biologique)
predictions = model.predict(dataset_test)[0]

print(f"\n--- PROFIL DE TOXICITÉ : {molecule} ---")
for index, nom_cible in enumerate(tasks):
    # L'indice [1] correspond à la probabilité d'être Toxique pour cette cible
    probabilite = predictions[index][1] 
    
    # On affiche une alerte si la probabilité dépasse 50%
    if probabilite > 0.5:
        print(f"⚠️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
    else:
        print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")


--- PROFIL DE TOXICITÉ : C#N ---
  Sûr    sur NR-AR      : 16.2%
  Sûr    sur NR-AR-LBD  : 12.1%
⚠️ DANGER sur NR-AhR     : 52.2%
  Sûr    sur NR-Aromatase : 7.8%
  Sûr    sur NR-ER      : 21.0%
  Sûr    sur NR-ER-LBD  : 12.7%
  Sûr    sur NR-PPAR-gamma : 37.0%
  Sûr    sur SR-ARE     : 41.4%
  Sûr    sur SR-ATAD5   : 32.6%
⚠️ DANGER sur SR-HSE     : 55.2%
  Sûr    sur SR-MMP     : 12.2%
  Sûr    sur SR-p53     : 14.2%


In [17]:
# 7. TESTONS LE PARACÉTAMOL ET LE VALDÉCOXIB
smiles_a_tester = {
    "Paracétamol": "CC(=O)NC1=CC=C(O)C=C1",
    "Valdécoxib": "CC1=C(C(=NO1)C2=CC=CC=C2)C3=CC=C(C=C3)S(=O)(=O)N"
}

noms = list(smiles_a_tester.keys())
smiles = list(smiles_a_tester.values())

graphes_medocs = featurizer.featurize(smiles)
dataset_medocs = dc.data.NumpyDataset(X=graphes_medocs)

predictions_medocs = model.predict(dataset_medocs)

for i, nom_molecule in enumerate(noms):
    print(f"\n=== PROFIL DE TOXICITÉ : {nom_molecule} ===")
    preds = predictions_medocs[i]
    for index, nom_cible in enumerate(tasks):
        probabilite = preds[index][1]
        if probabilite > 0.5:
            print(f"⚠️ DANGER sur {nom_cible:10s} : {probabilite*100:.1f}%")
        else:
            print(f"  Sûr    sur {nom_cible:10s} : {probabilite*100:.1f}%")


=== PROFIL DE TOXICITÉ : Paracétamol ===
  Sûr    sur NR-AR      : 44.3%
  Sûr    sur NR-AR-LBD  : 20.1%
⚠️ DANGER sur NR-AhR     : 68.2%
  Sûr    sur NR-Aromatase : 25.0%
⚠️ DANGER sur NR-ER      : 60.3%
⚠️ DANGER sur NR-ER-LBD  : 65.8%
  Sûr    sur NR-PPAR-gamma : 13.1%
⚠️ DANGER sur SR-ARE     : 61.9%
  Sûr    sur SR-ATAD5   : 43.4%
⚠️ DANGER sur SR-HSE     : 50.3%
⚠️ DANGER sur SR-MMP     : 69.5%
⚠️ DANGER sur SR-p53     : 54.8%

=== PROFIL DE TOXICITÉ : Valdécoxib ===
⚠️ DANGER sur NR-AR      : 70.7%
  Sûr    sur NR-AR-LBD  : 16.9%
⚠️ DANGER sur NR-AhR     : 71.1%
  Sûr    sur NR-Aromatase : 45.5%
  Sûr    sur NR-ER      : 44.2%
  Sûr    sur NR-ER-LBD  : 11.1%
  Sûr    sur NR-PPAR-gamma : 20.0%
⚠️ DANGER sur SR-ARE     : 72.7%
  Sûr    sur SR-ATAD5   : 31.2%
  Sûr    sur SR-HSE     : 7.4%
  Sûr    sur SR-MMP     : 33.2%
  Sûr    sur SR-p53     : 9.7%


### 🧬 Annexe : Explication des 12 cibles de toxicité (Tox21)
Le dataset Tox21 analyse la toxicité selon 12 voies biologiques précises. Elles sont divisées en deux grandes catégories :

#### 1. Récepteurs Nucléaires (NR - Perturbateurs Endocriniens)
Ces cibles vérifient si la molécule agit comme un perturbateur hormonal en se fixant sur nos récepteurs.

> **💡 Petit rappel de biologie :**
> * **Une Hormone :** C'est un messager chimique naturel de votre corps (ex: la testostérone, l'insuline). Elle voyage dans le sang et agit comme une "clé" qui rentre dans une "serrure" (le récepteur cellulaire) pour déclencher une action (grandir, stocker du sucre, etc.).
> * **Un Perturbateur Endocrinien :** C'est une molécule chimique artificielle qui a une forme tellement proche d'une vraie hormone qu'elle arrive à pirater la serrure de la cellule. Elle peut soit bloquer la serrure (empêchant la vraie hormone d'agir), soit l'activer au mauvais moment. Cela cause des problèmes de croissance, de fertilité, ou des cancers hormonodépendants.

* **`NR-AR` / `NR-AR-LBD`** : Récepteur aux Androgènes (Hormones masculines comme la testostérone). 
* **`NR-ER` / `NR-ER-LBD`** : Récepteur aux Œstrogènes (Hormones féminines). Souvent lié aux problèmes de fertilité.
* **`NR-Aromatase`** : L'enzyme qui convertit la testostérone en œstrogène.
* **`NR-PPAR-gamma`** : Régulateur du métabolisme des graisses et du sucre (souvent lié au diabète).
* **`NR-AhR`** : Récepteur aux Hydrocarbures (Le récepteur qui réagit aux dioxines et aux polluants chimiques).
  > **Quel est le rôle naturel du AhR dans notre corps ?** 
  > Le récepteur AhR n'est pas une hormone classique. C'est le **"détecteur de poison" naturel** de nos cellules. Son vrai rôle (forgé par l'évolution) est de surveiller notre environnement. S'il détecte des toxines naturelles (par exemple après avoir mangé une plante toxique), il s'active et ordonne à la cellule de produire des enzymes pour détruire et "nettoyer" le poison. 
  > **Le problème moderne :** Les polluants industriels artificiels (comme la dioxine, les pesticides, ou les fumées de diesel) ont des formes géométriques parfaites pour se coincer dans ce récepteur de façon permanente. Le détecteur panique et reste bloqué sur "ON". Le système se dérègle complètement, déclenchant une inflammation chronique massive et de très nombreux cancers.

#### 2. Réponses au Stress Cellulaire (SR - Dégâts et Cancers)
Ces cibles vérifient si la molécule attaque littéralement la machinerie interne de la cellule :
* **`SR-p53`** : La protéine p53 est le "Gardien du Génome". Si elle s'active, cela signifie que la molécule a causé des **dégâts majeurs à l'ADN** (risque très élevé de cancer).
* **`SR-ATAD5`** : Un autre marqueur majeur de génotoxicité (mutations de l'ADN).
* **`SR-MMP`** : Potentiel de la Membrane Mitochondriale. Si la molécule perturbe cela, elle détruit l'"usine à énergie" de la cellule et provoque sa mort (apoptose).
* **`SR-ARE`** : Réponse au stress oxydatif (les fameux radicaux libres).
* **`SR-HSE`** : Réponse aux chocs thermiques et aux protéines mal formées.